# TimesFM VN30 OHLC Volatility Comparison Test

Based on G7 Paper: Do extreme range estimators improve realized volatility forecasts?

Expected: 10-25% improvement with overnight volatility feature

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"GPU Memory: {gpu_mem:.1f} GB")
    if gpu_mem > 20:
        print("G4/L4 GPU detected - Perfect for TimesFM 2.5!")

In [ ]:
# Clone repository and navigate (FIXED: use os.chdir for persistence)
!git clone https://github.com/ntquy9901/stockvoli-research.git

import os
os.chdir('stockvoli-research')
!git pull origin master
!ls -la

# CRITICAL: Create necessary directories (same as G4 notebook)
!mkdir -p experiments models

print(f"Working directory: {os.getcwd()}")
print("Directories created: experiments, models")

## Install Dependencies (CRITICAL: Fix torchao conflict)

In [ ]:
# CRITICAL: Uninstall problematic torchao first (version conflict fix)
# Error: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported
!pip uninstall -y torchao

# Setup memory optimization (same as G4 notebook)
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print("Torchao uninstalled to fix version conflict (same as G4 notebook)")
print("Memory optimization enabled")
print("")
print("If you still get import errors, uncomment and run:")
print("# !pip install -q transformers peft torch pandas numpy pyyaml accelerate tqdm")
print("# !pip install --upgrade transformers")

In [ ]:
# Verify imports (test that all packages work)
import transformers
import peft
import torch
import pandas as pd
import numpy as np
import yaml
from tqdm import tqdm

print("All packages imported successfully!")
print(f"transformers: {transformers.__version__}")
print(f"torch: {torch.__version__}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

In [ ]:
# Load and verify config (same as G4 notebook)
import yaml
from pathlib import Path

with open('configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Config loaded successfully!")
print("\nG4/L4 Optimized Settings:")
print(f"  - Context length: {config['dataset']['context_length']}")
print(f"  - Batch size: {config['training']['batch_size']}")
print(f"  - Gradient accumulation: {config['training']['gradient_accumulation_steps']}")
print(f"  - Workers: {config['training']['num_workers']}")
print(f"  - Learning rate: {config['training']['learning_rate']}")
print(f"  - Optimizer: {config['training']['optimizer']}")
print(f"  - Epochs: {config['training']['num_epochs']} (will override to 2)")

print("\nTest Configuration:")
print("  - Baseline: RV_20 feature (2 epochs)")
print("  - New Feature: Overnight volatility (2 epochs)")
print("  - Expected time: ~7.4 hours total")
print("  - Expected improvement: 10-25% over baseline")

In [ ]:
# Verify data and OHLC features
import pandas as pd
from pathlib import Path

# Verify data exists
data_dir = Path('data/processed')
if data_dir.exists():
    files = list(data_dir.glob('*_processed.csv'))
    print(f"Data files: {len(files)} stocks found")
    
    # Check sample file for OHLC features
    sample_file = files[0]
    df = pd.read_csv(sample_file)
    print(f"\nSample: {sample_file.name}")
    print(f"Shape: {df.shape}")
    
    # Check OHLC features
    ohlc_features = ['overnight', 'parkinson', 'gk', 'close_to_close']
    print(f"\nOHLC Features:")
    for feat in ohlc_features:
        if feat in df.columns:
            count = df[feat].notna().sum()
            print(f"  {feat}: {count} observations")
else:
    print(f"\nData directory not found: {data_dir}")

In [ ]:
# Run the OHLC comparison test
print("="*70)
print("TimesFM VN30 OHLC Volatility Comparison Test")
print("="*70)
print("")
print("Testing:")
print("  1. RV_20 (baseline) - 2 epochs (~3.7 hours)")
print("  2. Overnight volatility - 2 epochs (~3.7 hours)")
print("")
print("Expected Results (based on G7 paper):")
print("  - Conservative: 10-15% improvement")
print("  - Optimistic: 15-25% improvement (Vietnamese TET effects)")
print("")
print("Start time:", pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'))
print("="*70)
print("")

# Run the test
!python src/quick_2epoch_test.py

## Monitor Training Progress (Optional - Run while training)

In [ ]:
# Monitor training progress (same as G4 notebook)
print("Training logs:")
!tail -20 experiments/quick_comparison.log

print("\nGPU utilization:")
!nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv

# Check GPU memory usage
import torch
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nGPU Memory: {allocated:.1f}/{total:.1f} GB ({allocated/total*100:.1f}% utilization)")

In [ ]:
# Check final results
import json
from pathlib import Path

results_file = Path('experiments/feature_comparison_results_fixed.json')

if results_file.exists():
    with open(results_file, 'r') as f:
        results = json.load(f)
    
    print("\n" + "="*70)
    print("FINAL RESULTS - TimesFM VN30 OHLC Comparison")
    print("="*70)
    
    # Extract results
    if 'RV_20' in results and 'overnight' in results:
        rv20_result = results['RV_20']
        overnight_result = results['overnight']
        
        if rv20_result['status'] == 'success' and overnight_result['status'] == 'success':
            rv20_loss = rv20_result['best_val_loss']
            overnight_loss = overnight_result['best_val_loss']
            improvement = (rv20_loss - overnight_loss) / rv20_loss * 100
            
            print(f"\nPerformance Comparison:")
            print(f"  RV_20 (baseline):      {rv20_loss:.6f}")
            print(f"  Overnight volatility:   {overnight_loss:.6f}")
            print(f"  Improvement:            {improvement:+.1f}%")
            print("")
            
            if improvement > 0:
                print("SUCCESS: Overnight volatility is better!")
                print(f"Improved forecasting by {improvement:.1f}% over baseline")
                print(f"G7 paper findings validated on Vietnamese market!")
                
                if improvement >= 10:
                    print(f"EXCEEDS CONSERVATIVE ESTIMATE (10%)!")
                if improvement >= 15:
                    print(f"APPROACHES OPTIMISTIC ESTIMATE (15-25%)!")
                    
                print("\nNext Steps:")
                print("  - Deploy overnight volatility to production")
                print("  - Test other OHLC features (Parkinson, Garman-Klass)")
                print("  - Build ensemble model for 25-35% improvement")
            else:
                print("BASELINE RV_20 IS STILL BETTER")
                print(f"Overnight volatility underperformed by {abs(improvement):.1f}%")
                print("\nNeed to investigate:")
                print("  - Why G7 findings don't apply to Vietnam")
                print("  - Check data quality and OHLC calculations")
                print("  - Market-specific research needed")
                
        else:
            print("\nTest failed for one or both features")
            print(f"RV_20 status: {rv20_result['status']}")
            print(f"Overnight status: {overnight_result['status']}")
            
            if 'error' in rv20_result:
                print(f"RV_20 error: {rv20_result['error']}")
            if 'error' in overnight_result:
                print(f"Overnight error: {overnight_result['error']}")
    else:
        print("\nResults incomplete")
        print(f"Available features: {list(results.keys())}")
        
    print("="*70)
else:
    print(f"\nResults file not found: {results_file}")
    print("Test may still be running or failed to complete")
    print("Check logs: experiments/quick_comparison.log")

## Summary & Expected Outcomes**Based on G7 Paper Findings:**- Tested across 7 G7 markets (US, UK, Japan, Germany, France, Italy, Canada)- Overnight volatility most consistent performer- Parkinson and Garman-Klass also effective**Conservative Expectations (70% confidence):**- 10-15% improvement over baseline RV_20- Better capture of weekend/holiday effects- More stable volatility predictions**Optimistic Expectations (30% confidence):**- 15-25% improvement (Vietnamese TET holiday effects)- Foundation for ensemble approach- Production-ready deployment**Technical Success (95% confidence):**- Training completes without GPU errors- Both features produce valid predictions- Reproducible experimental framework validated